## Библиотеки

In [ ]:
from __future__ import annotations
import os, re, io, json, math, csv, itertools, mimetypes
from pathlib import Path
from typing import Dict, List, Tuple, Optional

def safe_import(name):
    try:
        return __import__(name)
    except Exception:
        return None

PyPDF2 = safe_import('PyPDF2')
pdfminer = safe_import('pdfminer')
docx = safe_import('docx')
bs4 = safe_import('bs4')
pandas = safe_import('pandas')
PIL = safe_import('PIL')
pytesseract = safe_import('pytesseract')
chardet = safe_import('chardet')

## Папки

In [ ]:
SOURCE_DIR = "ПДнDataset/share"
INCLUDE_EXTS = {'doc','docx','gif','html','ipynb','jpeg','jpg','pdf','php','png','rtf','xls'}


## Кодировки

In [ ]:
def detect_encoding(raw_bytes: bytes) -> str:
    if chardet is None:
        return 'utf-8'
    try:
        res = chardet.detect(raw_bytes)
        enc = res.get('encoding') or 'utf-8'
        return enc
    except Exception:
        return 'utf-8'

## Регулярки

In [ ]:
# персональные данные
EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?:(?:\+7|8)\s*\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2})")
FIO_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+)?\b")
DOB_RE = re.compile(r"\b(\d{2}[./]\d{2}[./]\d{4})\b")
INDEX_RE = re.compile(r"\b\d{6}\b")

# Гос. идентификаторы
SNILS_RE = re.compile(r"\b\d{3}-\d{3}-\d{3}\s?\d{2}\b")
INN10_RE = re.compile(r"(?<!\d)\d{10}(?!\d)")
INN12_RE = re.compile(r"(?<!\d)\d{12}(?!\d)")
PASSPORT_RE = re.compile(r"(?:(?<!\d)\d{2}\s?\d{2}\s?\d{6}(?!\d))")
MRZ_RE = re.compile(r"[P|V|C]<[A-Z<]{2}")
DL_RE = re.compile(r"(?<!\d)\d{10,12}(?!\d)")  # водительское удостоверение

# Платёжные
CARD_RE = re.compile(r"(?:(?:\d[ -]*?){13,19})")
CVV_RE = re.compile(r"\b(CVV|CVC|CVV2)\b", re.IGNORECASE)
RS_RE = re.compile(r"(?i)(?:р/с|расч[её]тн(?:ый)?\s+сч[её]т)[^\d]*(\d{20})")
BIK_RE = re.compile(r"(?i)бик[^\d]*(\d{9})")

In [ ]:
def extract_from_text(text: str) -> dict:
    found = defaultdict(set)
    for name, pattern in PATTERNS.items():
        for match in pattern.finditer(text):
            value = match.group().strip()
            if name == "RS" and match.lastindex:
                value = match.group(1);
            elif name == "BIK" and match.lastindex:
                value == match.group(1)
            found[name].add(value)
    return {k: list(v) for k, v in found.items()}

def process_folder(folder_path: str) -> dict:
    """Обходит все текстовые файлы в папке и извлекает данные"""
    all_results = defaultdict(lambda: defaultdict(set))  # файл -> категория -> set значений
    folder = Path(folder_path)
    
    if not folder.exists():
        raise FileNotFoundError(f"Папка не найдена: {folder_path}")
    
    for file_path in folder.rglob("*"):
        if file_path.is_file() and is_text_file(file_path):
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
            except UnicodeDecodeError:
                # Пробуем другие кодировки
                try:
                    with open(file_path, 'r', encoding='cp1251') as f:
                        content = f.read()
                except Exception as e:
                    print(f"⚠️ Не удалось прочитать {file_path}: {e}")
                    continue
            except Exception as e:
                print(f"⚠️ Ошибка при чтении {file_path}: {e}")
                continue
            
            extracted = extract_from_text(content)
            if any(extracted.values()):  # если что-то нашли
                rel_path = str(file_path.relative_to(folder))
                for category, values in extracted.items():
                    all_results[rel_path][category].update(values)
                print(f"✓ Обработан {rel_path}")
    
    # Преобразуем вложенные set в list для JSON
    json_ready = {}
    for file_path, categories in all_results.items():
        json_ready[file_path] = {cat: list(vals) for cat, vals in categories.items()}
    return json_ready